In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ultranest

In [ ]:
data = []
with open("data.txt") as f:
    lines = f.readlines()[36:]
    for line in lines:
        if line[61] == "#":
            continue
        el = line[20:22].strip()
        N = int(line[6:9].strip())
        Z = int(line[11:14].strip())
        A = int(line[16:19].strip())
        binding_energy = float(line[56:67].strip()) * A / 1000
        binding_energy_std = float(line[68:77].strip()) * A / 1000
        if binding_energy_std < 1e-9:
            continue
        data.append([el, N, Z, A, binding_energy, binding_energy_std])
df = pd.DataFrame(data, columns=["Element", "N", "Z", "A", "Binding Energy (MeV)", "Binding Energy std (MeV)"])

In [ ]:
df

In [ ]:
def model_simple(N, Z, A, a_v, a_s, a_c, a_a):
    return a_v*A - a_s*A**(2/3)-a_c*Z*(Z-1)/A**(1/3)-a_a*(N-Z)**2/A

def prior_transform_simple(cube):
    params = np.zeros_like(cube)

    a_v_min = 10
    a_v_max = 20
    params[0] = cube[0] * (a_v_max - a_v_min) + a_v_min

    a_s_min = 15
    a_s_max = 25
    params[1] = cube[1] * (a_s_max - a_s_min) + a_s_min

    a_c_min = 0.1
    a_c_max = 1
    params[2] = cube[2] * (a_c_max - a_c_min) + a_c_min

    a_a_min = 15
    a_a_max = 25
    params[3] = cube[3] * (a_a_max - a_a_min) + a_a_min

    return params

def likelihood_simple(params):
    model = model_simple(df["N"], df["Z"], df["A"].values, *params)

    residual = (df["Binding Energy (MeV)"] - model) / df["Binding Energy std (MeV)"]

    log_norm = np.sum(np.log(2 * np.pi * df["Binding Energy std (MeV)"]**2))

    return -0.5 * (np.sum(residual**2) + log_norm)

params_simple = ["a_v", "a_s", "a_c", "a_a"]

In [ ]:
sampler_simple = ultranest.ReactiveNestedSampler(
    params_simple, likelihood_simple, prior_transform_simple
)
result_simple = sampler_simple.run()
sampler_simple.print_results()

In [ ]:
A_range = np.arange(np.min(df["A"]), np.max(df["A"]))

b = np.zeros_like(A)

for i, A in enumerate(A_range):
    for N in range(0, A + 1):
        Z = A - N
        en = model_simple(N, Z, A, *result_simple["posterior"]["mean"]) / A
        b[i] = np.max(b[i], en)

plt.plot(A_range, b, linestyle='', marker='.', label="Modello semplice")

plt.errorbar(df["A"], df["Binding Energy (MeV)"] / df["A"], df["Binding Energy std (MeV)"] / df["A"], label="Dati")

plt.legend()
plt.plot()

In [ ]:
def model_pairing(N, Z, A, a_v, a_s, a_c, a_a, a_p):
    delta_0 = a_p * A**(-1/2)
    # (A % 2 == 1) * ((N % 2 == 0) * 2 - 1)
    b = a_v * A \
        - a_s * A**(2/3) \
        - a_c * Z * (Z - 1) / A**(1/3) \
        - a_a * (N - Z)**2 / A \
        + np.where(A % 2 == 1, 0, np.where(N % 2 == 0, 1, -1)) * delta_0
    return b

def prior_transform_pairing(cube):
    params = np.zeros_like(cube)

    a_v_min = 0.1
    a_v_max = 20.0
    params[0] = 10**(cube[0] * (np.log10(a_v_max) - np.log10(a_v_min)) + np.log10(a_v_min))

    a_s_min = 0.1
    a_s_max = 20.0
    params[1] = 10**(cube[1] * (np.log10(a_s_max) - np.log10(a_s_min)) + np.log10(a_s_min))

    a_c_min = 0.1
    a_c_max = 20.0
    params[2] = 10**(cube[2] * (np.log10(a_c_max) - np.log10(a_c_min)) + np.log10(a_c_min))

    a_a_min = 0.1
    a_a_max = 20.0
    params[3] = 10**(cube[3] * (np.log10(a_a_max) - np.log10(a_a_min)) + np.log10(a_a_min))

    a_p_min = 5
    a_p_max = 40
    params[4] = 10**(cube[4] * (np.log10(a_p_max) - np.log10(a_p_min)) + np.log10(a_p_min))

    return params

def likelihood_pairing(params):
    model = model_pairing(df["N"], df["Z"], df["A"].values, *params)

    residual = (df["Binding Energy (MeV)"] - model) / (df["Binding Energy std (MeV)"] + 1e-9)

    log_norm = np.sum(np.log(2 * np.pi * (df["Binding Energy std (MeV)"] + 1e-9)**2))

    return -0.5 * (np.sum(residual**2) + log_norm)

params_pairing = ["a_v", "a_s", "a_c", "a_a", "a_p"]

In [ ]:
sampler_pairing = ultranest.ReactiveNestedSampler(
    params_pairing, likelihood_pairing, prior_transform_pairing
)
result_pairing = sampler_pairing.run()
sampler_pairing.print_results()